# Calculation of log-normal distributions

In [1]:
from scipy.stats import lognorm
import numpy as np

def lognormal_bin_edges(mu, sigma, edges):
    retvals = []
    for i, edge in enumerate(edges):
        if i == 0:
            cdf_min = lognorm.cdf(0, s=sigma, scale=np.exp(mu))
        else:
            cdf_min = lognorm.cdf(edges[i-1], s=sigma, scale=np.exp(mu))
        cdf_max = lognorm.cdf(edges[i], s=sigma, scale=np.exp(mu))
        retvals.append(cdf_max - cdf_min)
    return retvals

def truncated_lognormal_bin_edges(mu, sigma, x_min, x_max, n_bins):
    cdf_min = lognorm.cdf(x_min, s=sigma, scale=np.exp(mu))
    cdf_max = lognorm.cdf(x_max, s=sigma, scale=np.exp(mu))
    inner_probabilities = cdf_min + (cdf_max - cdf_min) * np.arange(1, n_bins) / n_bins
    inner_edges = lognorm.ppf(inner_probabilities, s=sigma, scale=np.exp(mu))
    return [x_min, *inner_edges, x_max]

def truncated_lognormal_bin_means(mu, sigma, bin_edges):
    means = [
        lognorm.expect(
            lambda x: x,
            args=(sigma,),
            scale=np.exp(mu),
            lb=left,
            ub=right,
            conditional=True,
        )
        for left, right in zip(bin_edges[:-1], bin_edges[1:])
    ]
    return np.array(means)

# bin edges where the weighted sum of two lognormal CDFs steps by equal increments
def bimodal_truncated_lognormal_bin_edges(mu1, sigma1, mu2, sigma2, w1, w2, x_min, x_max, n_bins):
    from scipy.optimize import brentq
    def cdf(x):
        return w1 * lognorm.cdf(x, s=sigma1, scale=np.exp(mu1)) \
             + w2 * lognorm.cdf(x, s=sigma2, scale=np.exp(mu2))
    cdf_min = cdf(x_min)
    cdf_max = cdf(x_max)
    edges = [x_min]
    for i in range(1, n_bins):
        target = cdf_min + i * (cdf_max - cdf_min) / n_bins
        edges.append(brentq(lambda x, t=target: cdf(x) - t, x_min, x_max))
    edges.append(x_max)
    return edges

def bimodal_truncated_lognormal_bin_means(mu1, sigma1, mu2, sigma2, w1, w2, bin_edges):
    means = []
    for left, right in zip(bin_edges[:-1], bin_edges[1:]):
        mass1 = lognorm.expect(lambda x: x, args=(sigma1,), scale=np.exp(mu1), lb=left, ub=right, conditional=False)
        mass2 = lognorm.expect(lambda x: x, args=(sigma2,), scale=np.exp(mu2), lb=left, ub=right, conditional=False)
        prob1 = lognorm.cdf(right, s=sigma1, scale=np.exp(mu1)) - lognorm.cdf(left, s=sigma1, scale=np.exp(mu1))
        prob2 = lognorm.cdf(right, s=sigma2, scale=np.exp(mu2)) - lognorm.cdf(left, s=sigma2, scale=np.exp(mu2))
        means_temp = (w1 * mass1 + w2 * mass2) / (w1 * prob1 + w2 * prob2)
        if np.isnan(means_temp):
            # This can happen if both prob1 and prob2 are zero, which can occur if the bin is very narrow and far in the tail of the distribution.
            # In this case, we can approximate the mean as the midpoint of the bin edges.
            means_temp = (left + right) / 2
        means.append(means_temp)
    return np.array(means)



## SNAP particle classes

In [2]:
# upper size, radius in µm
cutoff_radii = [3.0, 6.5, 11.5, 18.5, 29.0, 45.0, 71.0, 120.0, 250.0, 500.0]

## Glasstone-Dolan
fitted manually to ~(mu=3.78, sigma=0.68)

In [3]:
gd_mu = 3.78
gd_sigma = 0.68
gd = np.array(lognormal_bin_edges(gd_mu, gd_sigma, cutoff_radii))
np.set_printoptions(precision=2, floatmode='fixed', suppress=True)
display(gd, gd.sum())


array([0.00, 0.00, 0.02, 0.08, 0.17, 0.24, 0.25, 0.17, 0.06, 0.01])

np.float64(0.9998284039842542)

## Izrael

µ=4.04, sigma=1.68



In [4]:
izrael_mu = 4.04
izrael_sigma = 1.68
izrael = np.array(lognormal_bin_edges(izrael_mu, izrael_sigma, cutoff_radii))
display(izrael, izrael.sum())

display(truncated_lognormal_bin_means(izrael_mu, izrael_sigma, truncated_lognormal_bin_edges(izrael_mu, izrael_sigma, .0, 1000.0, 50)))
cutoff_radii_truncated = truncated_lognormal_bin_edges(izrael_mu, izrael_sigma, .0, 1000.0, 50)
izrael_even_bins = np.array(lognormal_bin_edges(izrael_mu, izrael_sigma, cutoff_radii_truncated))
display(izrael_even_bins, izrael_even_bins.sum())


array([0.04, 0.06, 0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.14, 0.09])

np.float64(0.9022380087560673)

array([  1.07,   2.33,   3.46,   4.58,   5.73,   6.93,   8.18,   9.50,
        10.88,  12.34,  13.89,  15.52,  17.26,  19.10,  21.06,  23.14,
        25.36,  27.73,  30.26,  32.97,  35.86,  38.97,  42.31,  45.90,
        49.77,  53.95,  58.48,  63.38,  68.72,  74.54,  80.91,  87.89,
        95.58, 104.08, 113.52, 124.04, 135.85, 149.17, 164.30, 181.63,
       201.66, 225.07, 252.75, 286.01, 326.73, 377.76, 443.72, 532.56,
       659.55, 858.76])

array([0.00, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02])

np.float64(0.9560887126443172)

## Bomb particles
below 20µm, using distribution µ=1, σ=0.8

In [5]:
bomb_sigma = 0.8
bomb_mu = 1

bomb = np.array(lognormal_bin_edges(bomb_mu, bomb_sigma, cutoff_radii))
display(bomb, bomb.sum())

izrael_bomb_even_bins = np.array(lognormal_bin_edges(bomb_mu, bomb_sigma, cutoff_radii_truncated))
display(izrael_bomb_even_bins, izrael_bomb_even_bins.sum())


array([0.55, 0.31, 0.10, 0.03, 0.01, 0.00, 0.00, 0.00, 0.00, 0.00])

np.float64(0.9999999999644362)

array([0.00, 0.29, 0.24, 0.16, 0.10, 0.07, 0.04, 0.03, 0.02, 0.01, 0.01,
       0.01, 0.01, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
       0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
       0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
       0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00])

np.float64(0.9999999999999236)